In [35]:
# Install required packages
# pip install xgboost pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

In [36]:
class DataPreprocessor:
    def __init__(self):
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.feature_names = None
        
    def load_data(self, file_path, target_column):
        """Load and prepare data"""
        # Load dataset
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        elif file_path.endswith('.xlsx'):
            df = pd.read_excel(file_path)
        else:
            raise ValueError("Unsupported file format")
            
        # Separate features and target
        X = df.drop(columns=[target_column])
        y = df[target_column]
        
        self.feature_names = X.columns.tolist()
        return X, y
    
    def handle_missing_values(self, X, strategy='mean'):
        """Handle missing values"""
        # If input is a numpy array, convert to DataFrame for missing value handling
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=self.feature_names)
        if strategy == 'mean':
            X = X.fillna(X.mean())
        elif strategy == 'median':
            X = X.fillna(X.median())
        elif strategy == 'mode':
            X = X.fillna(X.mode().iloc[0])
        elif strategy == 'drop':
            X = X.dropna()
        # If output should be numpy array (to match input), convert back
        if isinstance(X, pd.DataFrame):
            return X.values if isinstance(X, np.ndarray) else X
        return X
    
    def encode_categorical(self, X, y=None):
        """Encode categorical variables"""
        X_encoded = X.copy()
        
        for column in X_encoded.select_dtypes(include=['object']).columns:
            if column not in self.label_encoders:
                self.label_encoders[column] = LabelEncoder()
                X_encoded[column] = self.label_encoders[column].fit_transform(X_encoded[column].astype(str))
            else:
                X_encoded[column] = self.label_encoders[column].transform(X_encoded[column].astype(str))
        
        # Encode target if categorical
        if y is not None and hasattr(y, "dtype") and getattr(y, "dtype", None) == 'object':
            le_target = LabelEncoder()
            y_encoded = le_target.fit_transform(y)
            return X_encoded, y_encoded, le_target
        
        return X_encoded, y, None
    
    def scale_features(self, X, fit=True):
        """Scale numerical features"""
        if fit:
            X_scaled = self.scaler.fit_transform(X)
        else:
            X_scaled = self.scaler.transform(X)
        return X_scaled
    
    def prepare_data(self, X, y=None, is_training=True):
        """Complete data preparation pipeline"""
        # Handle missing values
        X_clean = self.handle_missing_values(X)
        
        # Encode categorical variables
        if y is not None:
            X_encoded, y_encoded, target_encoder = self.encode_categorical(X_clean, y)
        else:
            X_encoded, y_encoded, target_encoder = self.encode_categorical(X_clean)
        
        # Scale features
        if is_training:
            X_scaled = self.scale_features(X_encoded, fit=True)
        else:
            X_scaled = self.scale_features(X_encoded, fit=False)
        
        return X_scaled, y_encoded, target_encoder

In [45]:
from sklearn.metrics import mean_absolute_error
class XGBoostPipeline:
    def __init__(self, problem_type='classification'):
        self.problem_type = problem_type
        self.model = None
        self.preprocessor = DataPreprocessor()
        self.target_encoder = None
        self.best_params = None
        
    def create_model(self, **params):
        """Create XGBoost model based on problem type"""
        if self.problem_type == 'classification':
            self.model = xgb.XGBClassifier(**params)
        elif self.problem_type == 'regression':
            self.model = xgb.XGBRegressor(**params)
        else:
            raise ValueError("Problem type must be 'classification' or 'regression'")
    
    def hyperparameter_tuning(self, X_train, y_train, param_grid=None):
        """Perform hyperparameter tuning using GridSearchCV"""
        if param_grid is None:
            if self.problem_type == 'classification':
                param_grid = {
                    'n_estimators': [100, 200, 300],
                    'max_depth': [3, 5, 7],
                    'learning_rate': [0.01, 0.1, 0.2],
                    'subsample': [0.8, 0.9, 1.0],
                    'colsample_bytree': [0.8, 0.9, 1.0]
                }
            else:  # regression
                param_grid = {
                    'n_estimators': [100, 200, 300],
                    'max_depth': [3, 5, 7],
                    'learning_rate': [0.01, 0.1, 0.2],
                    'subsample': [0.8, 0.9, 1.0],
                    'colsample_bytree': [0.8, 0.9, 1.0]
                }
        
        grid_search = GridSearchCV(
            estimator=self.model,
            param_grid=param_grid,
            cv=5,
            scoring='accuracy' if self.problem_type == 'classification' else 'neg_mean_squared_error',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(X_train, y_train)
        self.best_params = grid_search.best_params_
        self.model = grid_search.best_estimator_
        
        print(f"Best parameters: {self.best_params}")
        print(f"Best score: {grid_search.best_score_:.4f}")
        
        return grid_search
    
    def train(self, X, y, tune_hyperparams=False, param_grid=None):
        """Train the XGBoost model"""
        # Prepare data
        X_processed, y_processed, target_encoder = self.preprocessor.prepare_data(X, y, is_training=True)
        self.target_encoder = target_encoder
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X_processed, y_processed, test_size=0.2, random_state=42, 
            stratify=y_processed if self.problem_type == 'classification' else None
        )
        
        # Create default model if not exists
        if self.model is None:
            self.create_model(random_state=42)
        
        # Hyperparameter tuning
        if tune_hyperparams:
            grid_search = self.hyperparameter_tuning(X_train, y_train, param_grid)
        else:
            # Train with current parameters
            self.model.fit(X_train, y_train)
        
        # Evaluate on test set
        test_predictions = self.predict(X_test, is_processed=True)
        
        if self.problem_type == 'classification':
            test_accuracy = accuracy_score(y_test, test_predictions)
            print(f"Test Accuracy: {test_accuracy:.4f}")
        else:
            test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
            print(f"Test RMSE: {test_rmse:.4f}")
        
        return X_train, X_test, y_train, y_test
    
    def predict(self, X, is_processed=False):
        """Make predictions"""
        if not is_processed:
            X_processed, _, _ = self.preprocessor.prepare_data(X, is_training=False)
        else:
            X_processed = X
        
        predictions = self.model.predict(X_processed)
        
        # Convert back to original labels for classification
        if self.problem_type == 'classification' and self.target_encoder is not None:
            predictions = self.target_encoder.inverse_transform(predictions)
        
        return predictions
    
    def predict_proba(self, X):
        """Get prediction probabilities (classification only)"""
        if self.problem_type != 'classification':
            raise ValueError("predict_proba is only available for classification problems")
        
        X_processed, _, _ = self.preprocessor.prepare_data(X, is_training=False)
        return self.model.predict_proba(X_processed)
    
    def evaluate(self, X, y):
        """Comprehensive model evaluation"""
        X_processed, y_processed, _ = self.preprocessor.prepare_data(X, y, is_training=False)
        predictions = self.predict(X, is_processed=False)
        
        if self.problem_type == 'classification':
            accuracy = accuracy_score(y, predictions)
            print(f"Accuracy: {accuracy:.4f}")
            print("\nClassification Report:")
            print(classification_report(y, predictions))
            
            # Confusion matrix
            plt.figure(figsize=(8, 6))
            cm = confusion_matrix(y, predictions)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title('Confusion Matrix')
            plt.show()
            
        else:
            mse = mean_squared_error(y, predictions)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y, predictions)
            
            print(f"MSE: {mse:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f"MAE: {mae:.4f}")
            
            # Residual plot
            # plt.figure(figsize=(10, 6))
            # residuals = y - predictions
            # plt.scatter(predictions, residuals, alpha=0.5)
            # plt.axhline(y=0, color='red', linestyle='--')
            # plt.xlabel('Predicted Values')
            # plt.ylabel('Residuals')
            # plt.title('Residual Plot')
            # plt.show()
    
    def feature_importance(self, top_n=10):
        """Plot feature importance"""
        if self.model is None:
            raise ValueError("Model not trained yet")
        
        importance_scores = self.model.feature_importances_
        feature_names = self.preprocessor.feature_names
        
        # Create feature importance DataFrame
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importance_scores
        }).sort_values('importance', ascending=False)
        
        # Plot
        # plt.figure(figsize=(10, 8))
        # sns.barplot(data=importance_df.head(top_n), x='importance', y='feature')
        # plt.title(f'Top {top_n} Feature Importance')
        # plt.tight_layout()
        # plt.show()
        
        return importance_df
    
    def save_model(self, filepath):
        """Save the entire pipeline"""
        pipeline = {
            'model': self.model,
            'preprocessor': self.preprocessor,
            'target_encoder': self.target_encoder,
            'problem_type': self.problem_type,
            'best_params': self.best_params
        }
        joblib.dump(pipeline, filepath)
        print(f"Pipeline saved to {filepath}")
    
    def load_model(self, filepath):
        """Load a saved pipeline"""
        pipeline = joblib.load(filepath)
        self.model = pipeline['model']
        self.preprocessor = pipeline['preprocessor']
        self.target_encoder = pipeline['target_encoder']
        self.problem_type = pipeline['problem_type']
        self.best_params = pipeline['best_params']
        print(f"Pipeline loaded from {filepath}")

In [46]:
Data = pd.read_csv('train.csv')  # Load training data

In [47]:
Data = Data.drop(columns=['id'])  # As 'id' is not a feature
Data = Data.fillna(Data.mean())  # Fill missing values with column means
y = Data['song_popularity'].copy()
y = pd.DataFrame({'song_popularity': y})
X = Data.drop(columns=['song_popularity'])

In [49]:
def balanced_dataset_creation(X, y, num, random_state=42):
    X_0 = X[y['song_popularity'] == 0]
    X_1 = X[y['song_popularity'] == 1]

    X_0_sampled = X_0.sample(n=num, random_state=random_state)
    X_1_sampled = X_1.sample(n=num, random_state=random_state)

    y_0 = pd.DataFrame({'song_popularity': [0]*num})
    y_1 = pd.DataFrame({'song_popularity': [1]*num})

    y = pd.concat([y_0, y_1], ignore_index=True)
    X = pd.concat([X_0_sampled, X_1_sampled], ignore_index=True)

    # shuffle the dataset
    index = list(range(2*num))
    random.Random(random_state).shuffle(index)
    X = X.iloc[index]
    y = y.iloc[index]

    return X, y

In [53]:
y = y.iloc[:, 0]

In [52]:
# Example with California housing dataset (regression)
def regression_example():
    from sklearn.datasets import fetch_california_housing
    from sklearn.metrics import mean_absolute_error
    
    # Load data
    data = fetch_california_housing()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(data.target, name='price')

    X = X[:100]
    y = y[:100]
    
    # Create and train pipeline
    pipeline = XGBoostPipeline(problem_type='regression')
    
    # Train
    X_train, X_test, y_train, y_test = pipeline.train(
        X, y,
        tune_hyperparams=True
    )
    
    # Evaluate
    pipeline.evaluate(X_test, y_test)
    
    # Feature importance
    pipeline.feature_importance()
    
    # Save model
    pipeline.save_model('xgboost_regression_pipeline.pkl')
    
    return pipeline

# Run regression example
regression_pipeline = regression_example()

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}
Best score: -0.3512
Test RMSE: 0.4980
MSE: 1.0010
RMSE: 1.0005
MAE: 0.8727
Pipeline saved to xgboost_regression_pipeline.pkl


In [54]:
def regression(X,y):
    from sklearn.metrics import mean_absolute_error
    
    # Create and train pipeline
    pipeline = XGBoostPipeline(problem_type='regression')
    
    # Train
    X_train, X_test, y_train, y_test = pipeline.train(
        X, y,
        tune_hyperparams=True
    )
    
    # Evaluate
    pipeline.evaluate(X_test, y_test)
    
    # Feature importance
    pipeline.feature_importance()
    
    # Save model
    pipeline.save_model('xgboost_regression_pipeline.pkl')
    
    return pipeline

# Run regression example
regression_pipeline = regression(X,y)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.9}
Best score: -0.2342
Test RMSE: 0.4709
MSE: 0.2191
RMSE: 0.4681
MAE: 0.4366
Pipeline saved to xgboost_regression_pipeline.pkl


In [ ]:
def cross_validate_model(pipeline, X, y, cv=5):
    """Perform cross-validation"""
    X_processed, y_processed, _ = pipeline.preprocessor.prepare_data(X, y, is_training=True)
    
    if pipeline.problem_type == 'classification':
        scoring = 'accuracy'
    else:
        scoring = 'neg_mean_squared_error'
    
    scores = cross_val_score(pipeline.model, X_processed, y_processed, cv=cv, scoring=scoring)
    
    if pipeline.problem_type == 'classification':
        print(f"Cross-validation Accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")
    else:
        rmse_scores = np.sqrt(-scores)
        print(f"Cross-validation RMSE: {rmse_scores.mean():.4f} (+/- {rmse_scores.std() * 2:.4f})")
    
    return scores

In [ ]:
def train_with_early_stopping(X, y, problem_type='classification'):
    """Train with early stopping"""
    # Prepare data
    preprocessor = DataPreprocessor()
    X_processed, y_processed, target_encoder = preprocessor.prepare_data(X, y, is_training=True)
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(X_processed, y_processed, test_size=0.2, random_state=42)
    
    # Create model with early stopping
    if problem_type == 'classification':
        model = xgb.XGBClassifier(
            n_estimators=1000,  # Large number
            early_stopping_rounds=50,
            random_state=42
        )
        eval_metric = 'logloss'
    else:
        model = xgb.XGBRegressor(
            n_estimators=1000,
            early_stopping_rounds=50,
            random_state=42
        )
        eval_metric = 'rmse'
    
    # Train with validation set
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    print(f"Best iteration: {model.best_iteration}")
    print(f"Best score: {model.best_score}")
    
    return model, preprocessor, target_encoder